In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

chrome_options = Options()
chrome_options.add_argument("--headless") 
chrome_options.add_argument("--start-maximized")

yetenek_sozlugu = {
    "Python": ["python"],
    "SQL": ["sql", "veritabanı", "database"],
    "Excel": ["excel", "ms office", "ofis programları"],
    "İngilizce": ["ingilizce", "english"],
    "İletişim": ["iletişim", "diksiyon", "ikna"],
    "Liderlik": ["liderlik", "yönetim", "sevk", "idare"],
    "Analiz": ["analiz", "raporlama", "reporting"],
    "Takım_Calısması": ["takım çalışması", "ekip çalışması"]
}

def ilan_detay_cek(link):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    try:
        res = requests.get(link, headers=headers, timeout=10)
        if res.status_code != 200: return None
        
        soup = BeautifulSoup(res.content, 'html.parser')
        
        baslik_etiket = soup.find(class_="u-text-ilan-baslik")
        baslik = baslik_etiket.text.strip() if baslik_etiket else "Başlık Bulunamadı"
        
        detay_etiket = soup.find(class_="d-information")
        detay_alani = detay_etiket.get_text().lower() if detay_etiket else ""

        bulunanlar = {y: (1 if any(k in detay_alani for k in kv) else 0) for y, kv in yetenek_sozlugu.items()}
        bulunanlar["Pozisyon"] = baslik
        bulunanlar["Link"] = link
        
        bulunanlar["Beceri_Sayisi"] = sum(1 for y in yetenek_sozlugu if bulunanlar[y] == 1)
        
        return bulunanlar
    except:
        return None

def eleman_net_hizli_kazi(sayfa_limiti=10):
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    ilan_linkleri = []

    print(f"{sayfa_limiti} sayfa taranıyor, linkler toplanıyor...")
    for sayfa in range(1, sayfa_limiti + 1):
        url = f"https://www.eleman.net/is-ilanlari?sy={sayfa}"
        driver.get(url)
        time.sleep(1.5) 
        
        linkler = driver.find_elements(By.CSS_SELECTOR, ".ilan_listeleme_bol > a")
        for link in linkler:
            l = link.get_attribute("href")
            if l not in ilan_linkleri:
                ilan_linkleri.append(l)
        
        if sayfa % 10 == 0: print(f"Sayfa {sayfa} bitti. Toplanan link: {len(ilan_linkleri)}")

    driver.quit()
    print(f"Toplam {len(ilan_linkleri)} ilan linki toplandı. Paralel işleme geçiliyor...")

    with ThreadPoolExecutor(max_workers=15) as executor:
        sonuclar = list(executor.map(ilan_detay_cek, ilan_linkleri))

    temiz_veriler = [s for s in sonuclar if s is not None]
    return pd.DataFrame(temiz_veriler)

df_eleman = eleman_net_hizli_kazi(sayfa_limiti=100) 

if not df_eleman.empty:
    df_eleman.to_csv("eleman_net_hizli_analiz.csv", index=False, encoding="utf-8-sig")
    print(f"\nİşlem bitti! {len(df_eleman)} ilan başarıyla analiz edildi.")
    print(df_eleman.head())
else:
    print("Veri toplanamadı.")